This notebook is for loading YOLO and RF-DETR models from the model.pt file. Afterwards, the model can be exported to ONNX. Inspect model graph and clear value_info for Qualcomm AI Hub compatibility. Upload onnx model to Qualcomm AI Hub. Compile for different runtimes in either FP32 or INT8. Profile model in Qualcomm AI Hub for metrics on inference speed and memory usage.

## Install dependencies

In [1]:
!pip install -q ultralytics roboflow onnx qai-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.0/122.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 47.4 MB/s eta 0:00:00


In [ ]:
!pip install -q rfdetr>=1.4.0

# Fixed package.
!pip uninstall -y pillow
!pip install -q pillow==10.4.0

Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 70.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pi-heif 1.3.0 requires pillow>=11.1.0, but you have pillow 10.4.0 which is incompatible.


## Imports

In [2]:
from google.colab import userdata, files
from IPython.display import Image
import onnx
import subprocess
import qai_hub as hub
from roboflow import Roboflow
import os
import glob
import cv2
import numpy as np

## Export Model for iOS App

For ios app deployment.

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/best.pt")
model.export(format="coreml", int8=False, imgsz=640, nms=False)

Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO26n-pose summary (fused): 132 layers, 3,825,678 parameters, 0 gradients, 11.7 GFLOPs

PyTorch: starting from '/content/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 105) (11.0 MB)

CoreML: starting export with coremltools 9.0...


Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 63.10 passes/s]


CoreML: export success ✅ 7.8s, saved as '/content/best.mlpackage' (7.6 MB)

Export complete (8.6s)
Results saved to /content/best.mlpackage
Predict:         yolo predict task=pose model=/content/best.mlpackage imgsz=640 
Validate:        yolo val task=pose model=/content/best.mlpackage imgsz=640 data=/content/datasets/basketball-court-detection-2-19/data.yaml  
Visualize:       https://netron.app


PosixPath('/content/best.mlpackage')

In [ ]:
from ultralytics.utils.downloads import zip_directory
zip_directory("/content/best.mlpackage").rename("yolo26n-posefp32.mlpackage.zip")

Deleting .DS_Store files: []
Deleting __MACOSX files: []
Zipping /content/best.mlpackage to /content/best.zip...: 100% ━━━━━━━━━━━━ 3/3 7.3files/s 0.4s


PosixPath('yolo26n-posefp32.mlpackage.zip')

## Export Model to ONNX (for Qualcomm AI Hub)
Supports YOLO26 models or RF-DETR models.

 For YOLO26 models:

In [ ]:
!yolo export model="/content/best.pt" imgsz=640 format=onnx opset=13


Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26s summary (fused): 122 layers, 9,468,663 parameters, 0 gradients, 20.5 GFLOPs

PyTorch: starting from '/content/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (19.4 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 257ms
Prepared 3 packages in 262ms
Installed 3 packages in 12ms
 + colorama==0.4.6
 + onnxruntime==1.26.0
 + onnxslim==0.1.93

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 13...
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/

In [ ]:
!yolo export task=pose model="/content/best.pt" imgsz=640 format=onnx opset=13


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26s-pose summary (fused): 132 layers, 11,581,494 parameters, 0 gradients, 29.2 GFLOPs

PyTorch: starting from '/content/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 105) (27.8 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 201ms
Prepared 3 packages in 279ms
Installed 3 packages in 13ms
 + c

For RF-DETR models:

In [ ]:
# Import the RF-DETR model you need
from rfdetr import RFDETRSmall

In [ ]:
model = RFDETRSmall(pretrain_weights="/content/checkpoint_best_total.pth")

[2026-05-21 12:03:52] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-21 12:03:52] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-21 12:03:53] [WARNING] rf-detr - Checkpoint has 10 classes but model is configured for 90. Using checkpoint class count (10). Pass num_classes=10 to suppress this warning.


In [ ]:
model.optimize_for_inference()

Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!


In [ ]:
model.export(format="onnx", opset=13, imgsz=640)

[2026-05-21 11:53:16] [INFO] rf-detr - Exporting model to ONNX format


Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!


[2026-05-21 11:53:21] [INFO] rf-detr - 
Successfully exported ONNX model: output/inference_model.onnx
[2026-05-21 11:53:21] [INFO] rf-detr - Successfully exported ONNX model to: output/inference_model.onnx
[2026-05-21 11:53:21] [INFO] rf-detr - ONNX export completed successfully


In [ ]:
!mv /content/output/inference_model.onnx /content/player-detector-rfdetr-small-B.onnx

In [ ]:
model_specs = "player-detector-rfdetr-small-B" # INPUT MODEL INFO
model_name = f"/content/{model_specs}.onnx"

Check model graph to see input and output specs (only for YOLO models)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/best.pt")

print(model.names)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{0: 'ball', 1: 'ball-in-basket', 2: 'player', 3: 'player-in-possession', 4: 'player-jump-shot', 5: 'player-layup-dunk', 6: 'player-shot-block', 7: 'referee', 8: 'rim'}


In [ ]:
model = onnx.load("/content/best.onnx")

# Completely remove ALL value_info
model.graph.ClearField("value_info")

model_specs = "court-detector-yolo26s-C" # INPUT MODEL INFO
model_name = f"/content/{model_specs}.onnx"
onnx.save(model, model_name)

In [ ]:
model = onnx.load(model_name)
onnx.checker.check_model(model)

print("Inputs:")
for tensor in model.graph.input:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    print(f" {tensor.name}: {dims}")

print("Outputs:")
output_shapes = {}
for tensor in model.graph.output:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    output_shapes[tensor.name] = dims
    print(f" {tensor.name}: {dims}")

Inputs:
 images: [1, 3, 640, 640]
Outputs:
 output0: [1, 300, 105]


## Upload to QAI-HUB
For model optimization for Edge devices. **Important** upload QAI_HUB_API_TOKEN to Google Colab Secrets.

In [ ]:
api_token = userdata.get('QAI_HUB_API_TOKEN')
_ = subprocess.run(
    [
        "qai-hub",
        "configure",
        "--api_token",
        api_token
    ],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

### Listing Available QAI Hub Devices

Before targeting a specific device for compilation and profiling, you can query the full list of devices supported by QAI Hub. The `qai-hub list-devices` command returns all currently available Snapdragon-powered devices — including smartphones, tablets, and compute platforms — that are accessible through QAI Hub's cloud infrastructure.

This helps identify supported device names to pass to `hub.Device()` and `client.submit_compile_job()` in subsequent steps.

In [ ]:
!qai-hub list-devices

+---------------------------------+--------------+----------+---------+---------------------------------------------------+------------------------------------------------------------+
|              Device             |      OS      |  Vendor  |   Type  |                      Chipset                      |                       CLI Invocation                       |
+---------------------------------+--------------+----------+---------+---------------------------------------------------+------------------------------------------------------------+
|     Google Pixel 3 (Family)     |  Android 10  |  Google  |  Phone  |          qualcomm-snapdragon-845, sdm845          |     --device "Google Pixel 3 (Family)" --device-os 10      |
|          Google Pixel 3         |  Android 10  |  Google  |  Phone  |          qualcomm-snapdragon-845, sdm845          |          --device "Google Pixel 3" --device-os 10          |
|         Google Pixel 3a         |  Android 10  |  Google  |  Phone  |    

## Converting the Model to QAI Hub DLC Format

QAI Hub compiles models to Qualcomm's **QNN DLC (Deep Learning Container)** format in the cloud, targeting a specific Snapdragon device. Unlike local SDK tools such as `snpe-onnx-to-dlc` or `qairt-converter`, the entire compilation pipeline runs remotely on Qualcomm's infrastructure — no local SDK installation is required.

The cell below initializes the QAI Hub client, selects the target device, and defines the output paths for the FP32 and INT8 DLC files.

In [ ]:
client = hub.Client()

DEVICE = hub.Device("Samsung Galaxy Tab S7")

model_onnx = model_name
fp32_tflite = f"/content/{model_specs}-fp32.tflite"
#FP32_ONNX = "../models/LibreYOLOXs.onnx"
#FP32_DLC = "../models/qaihub/LibreYOLOXs_fp32.dlc"
#INT8_DLC = "../models/qaihub/LibreYOLOXs_int8.dlc"

(Optional) get_model and compile + submit:

In [ ]:
client = hub.Client()
fp32_onnx_model = hub.get_model("mm6dwrg2q")
DEVICE = hub.Device("Samsung Galaxy S25+")
model_specs = "player-detector-rfdetr-small-B-SGS25+" # INPUT MODEL INFO

### Compiling the FP32 Model

The ONNX model is first uploaded to QAI Hub via `client.upload_model()`, then compiled to a **floating-point QNN DLC** by submitting a compile job. The job runs on Qualcomm's cloud infrastructure and targets the chosen Snapdragon device.

Compilation options specify:
- `--target_runtime qnn_dlc` — produces a QNN-compatible DLC for on-device inference.
- `--output_names bboxes,scores` — maps the restructured ONNX outputs to named outputs in the compiled DLC.

Once the job completes, the compiled FP32 DLC is downloaded locally for profiling.

In [ ]:
fp32_onnx_model = client.upload_model(model_onnx)

Uploading court-detector-yolo26s-C.onnx


100%|██████████| 44.5M/44.5M [00:02<00:00, 20.2MB/s]


(Optional) Use this for RF-DETR Models. They differ in input/output.

In [ ]:
compile_options = [
    "--target_runtime tflite",        # Qualcomm runtime
    "--output_names dets,labels"  # Custom output names
]
input_specs= {
 "input": (1, 3, 512, 512),
}

(Optional) Use this for YOLO Models. They differ in inpup/output

In [ ]:
compile_options = [
    "--target_runtime tflite",        # Qualcomm runtime
    "--output_names output0"  # Custom output names
]
input_specs= {
 "images": (1, 3, 640, 640),
}

In [ ]:
options_str = " ".join(compile_options)

compile_fp32_job = client.submit_compile_job(
    name=f"{model_specs}-fp32",
    model=fp32_onnx_model,
    device=DEVICE,
    options=options_str,
    input_specs=input_specs
)
compile_fp32_job.wait()

fp32_dlc_model = compile_fp32_job.get_target_model()
#fp32_dlc_model.download(fp32_tflite)
#files.download(fp32_tflite)

Scheduled compile job (jgledy4lp) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgledy4lp/

Waiting for compile job (jgledy4lp) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


### Profiling the FP32 Model

QAI Hub measures actual on-device inference latency by submitting a profile job that runs the compiled model on the target hardware. For the FP32 model, profiling is limited to `cpu` and `gpu` compute units — the HTP (Hexagon Tensor Processor) backend requires INT8 precision and is not compatible with floating-point DLCs.

The returned job object includes detailed per-layer and end-to-end timing results, which can be inspected directly in the notebook.

In [ ]:
profile_fp32_job = client.submit_profile_job(
    name=f"{model_specs}-fp32",
    model=fp32_dlc_model,
    device=DEVICE,
    options="--compute_unit cpu,gpu",
)
#profile_fp32_job.wait()
#profile_fp32_job

Scheduled profile job (jp3qdznz5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp3qdznz5/



### Preparing the Calibration Dataset - For INT8 quantize

QAI Hub's cloud quantization workflow (`client.submit_quantize_job()`) accepts calibration inputs as a dictionary of NumPy arrays keyed by input tensor name. These arrays are loaded from `.raw` binary files — flat binary files containing `float32` pixel values with shape `(C, H, W)`.

The code below:
1. Downloads the player-detection dataset A or B. Or court-detection dataset C.
2. Prepares the data and builds the calibration data
3. Afterwards, submit for quantize job

### Download the dataset
First download the dataset needed for building the calibration set for the model. Download the dataset which is required for the model that is in the process of being evaluated and submitted to Qualcomm AI Hub

In [ ]:
HOME = os.getcwd()
print(HOME)

/content


In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

/content/datasets


Dataset A (base classes):

In [ ]:
project = rf.workspace("roboflow-universe-projects").project("basketball-players-fy4c2")
version = project.version(25)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Basketball-Players-25 in yolo26:: 100%|██████████| 2404/2404 [00:00<00:00, 2622.44it/s]


Dataset B (with action classes):

In [ ]:
project = rf.workspace("roboflow-jvuqo").project("basketball-player-detection-3-ycjdo")
version = project.version(18)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to basketball-player-detection-3-18 in yolo26:: 100%|██████████| 1313/1313 [00:03<00:00, 381.27it/s]


Dataset C (court detection):

In [ ]:
project = rf.workspace("roboflow-jvuqo").project("basketball-court-detection-2")
version = project.version(19)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to basketball-court-detection-2-19 in yolo26:: 100%|██████████| 2932/2932 [00:00<00:00, 4818.17it/s]


### Build the calibration set

(Optional) YOLO img_sz

In [ ]:
IMG_SIZE = 640

(Optional) RF-DETR img_sz

In [ ]:
IMG_SIZE = 384

In [ ]:
dataset_path = dataset.location

# --------------------------------------------------
# FIND IMAGES
# --------------------------------------------------

image_paths = glob.glob(
    os.path.join(dataset_path, "train", "images", "*")
)

# Use representative calibration images
image_paths = image_paths[:100]

# --------------------------------------------------
# BUILD CALIBRATION DATA
# --------------------------------------------------

calibration_inputs = {
    "images": []
}

for path in image_paths:
    img = cv2.imread(path)

    if img is None:
        continue

    # Ultralytics YOLO preprocessing
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    img = img.astype(np.float32) / 255.0

    # HWC -> CHW
    img = np.transpose(img, (2, 0, 1))

    # Add batch dimension
    img = np.expand_dims(img, axis=0)

    calibration_inputs["images"].append(img)


print(f"Calibration samples: {len(calibration_inputs['images'])}")

Calibration samples: 100


(Optional) Rename key for expected input of RF-DETR Models


In [ ]:
calibration_inputs["input"] = calibration_inputs.pop("images")

### INT8 Post-Training Quantization and Compilation

Running inference in **8-bit integer (INT8)** precision is significantly faster and more energy-efficient on Qualcomm® hardware than FP32 — with only a small accuracy trade-off. QAI Hub performs quantization in the cloud via `client.submit_quantize_job()`, using the calibration `.raw` files loaded as NumPy arrays.

The calibration inputs are passed as a dictionary of NumPy arrays keyed by the input tensor name `"images"`. After quantization, the resulting model is compiled to a QNN DLC with INT8 precision and downloaded locally.

Compilation options specify:
- `--quantize_full_type int8` — applies full INT8 quantization to all weights and activations.
- `--quantize_io` — quantizes the model's input and output tensors.
- `--target_runtime qnn_dlc` and `--output_names bboxes,scores` — same as the FP32 compilation step.

In [ ]:
# --------------------------------------------------
# QUANTIZE MODEL
# --------------------------------------------------

quantize_job = hub.submit_quantize_job(
    name=f"{model_specs}-int8",
    model=fp32_onnx_model, # FP32_onnx_model
    calibration_data=calibration_inputs,
)

print("Waiting for quantization job...")
quantize_job.wait()

quantized_model = quantize_job.get_target_model()

Uploading dataset: 136MB [00:03, 40.2MB/s]                           


Scheduled quantize job (jgnrdo0r5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgnrdo0r5/

Waiting for quantization job...
Waiting for quantize job (jgnrdo0r5) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


(Optional) YOLO Compile Options

In [ ]:
compile_options = [
    "--target_runtime tflite",
    "--quantize_full_type int8",
    "--quantize_io",
    "--output_names output0"
]

(Optional) RF-DETR Compile Options.
Not quantizeable with current export.

In [ ]:
compile_options = [
    "--target_runtime tflite",
    "--quantize_full_type int8",
    "--quantize_io",
    "--truncate_64bit_tensors",
    "--output_names det,labels"
]

In [ ]:
options_str = " ".join(compile_options)

# --------------------------------------------------
# COMPILE MODEL
# --------------------------------------------------

compile_int8_job = hub.submit_compile_job(
    name=f"{model_specs}-int8",
    model=quantized_model,
    device=DEVICE,
    input_specs=input_specs,
    options=options_str,
)

print("Waiting for compile job...")
compile_int8_job.wait()

# --------------------------------------------------
# DOWNLOAD DLC
# --------------------------------------------------

int8_model = compile_int8_job.get_target_model()

int8_model.download(f"/content/{model_specs}-int8.tflite")
files.download(f"/content/{model_specs}-int8.tflite")
print("Done.")

Scheduled compile job (j56q9r20g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/j56q9r20g/

Waiting for compile job...
Waiting for compile job (j56q9r20g) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


court-detector-yolo26s-C-int8.tflite: 100%|██████████| 11.9M/11.9M [00:01<00:00, 10.1MB/s]

Downloaded model to /content/court-detector-yolo26s-C-int8.tflite


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.


### Profiling the INT8 Model

The INT8 DLC is profiled across all available compute units — including the **HTP (Hexagon Tensor Processor / DSP)** — using `--compute_unit all`. The HTP backend is specifically designed for INT8 inference and delivers the highest throughput and lowest power consumption on Qualcomm® Snapdragon chipsets.

Comparing the FP32 and INT8 profiling results shows the latency reduction and speedup achieved through quantization on the target device.

In [ ]:
profile_int8_job = client.submit_profile_job(
    name=f"{model_specs}-int8",
    model=int8_model,
    device=DEVICE,
    options="--compute_unit all",
)
profile_int8_job.wait()
profile_int8_job

Scheduled profile job (jp3qlxnl5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp3qlxnl5/

Waiting for profile job (jp3qlxnl5) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


ProfileJob
----------
job_id  : jp3qlxnl5
url     : https://workbench.aihub.qualcomm.com/jobs/jp3qlxnl5/
status  : SUCCESS
model   : Model(model_id='mnoyv79pm', name='job_j56q9r20g_optimized_tflite')
name    : court-detector-yolo26s-C-int8
options : --compute_unit all
shapes  : {'images': ((1, 3, 640, 640), 'int8')}
device  : Device(name='Samsung Galaxy Tab S7', os='11', attributes=['os:android', 'framework:tflite', 'framework:onnx', 'abi:aarch64-android', 'vendor:samsung', 'format:tablet', 'chipset:qualcomm-snapdragon-865+', 'chipset:sm8250-ab', 'hexagon:v66', 'soc-model:21'])
date    : 2026-05-21 14:35:47